<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); border-radius: 16px; padding: 48px 44px; margin-bottom: 8px;">
  <div style="display: flex; align-items: center; gap: 28px; font-family: system-ui, -apple-system, sans-serif;">
    <img src="../docs/assets/logo.png" alt="Gradients" style="height: 88px;">
    <div>
      <h1 style="color: #fff; margin: 0; font-size: 2.2em; letter-spacing: -0.01em;">Gradients</h1>
      <p style="color: #a8b2d1; margin: 10px 0 0 0; font-size: 1.15em; line-height: 1.5;">Post-training, simplified.<br>Fine-tune any model on your data with a single API call.</p>
    </div>
  </div>
</div>

<div style="border-left: 4px solid #7c3aed; padding: 20px 24px; margin: 36px 0 14px 0; background: linear-gradient(90deg, #f5f3ff 0%, #ffffff 100%); border-radius: 0 12px 12px 0; font-family: system-ui, -apple-system, sans-serif;">
  <div style="font-weight: 700; font-size: 1.2em; color: #1a1a2e;">The Problem: Base models are general-purpose</div>
  <div style="color: #6b7280; font-size: 0.95em; margin-top: 6px; line-height: 1.5;">Ask a base model a biomedical research question and you get vague hedging, hallucinated citations, or an answer that ignores the clinical evidence entirely. It doesn't know the format, the terminology, or how to reason from abstracts to a conclusion.<br><br>Here's what that looks like on a real question from <a href="https://huggingface.co/datasets/gradients-io-tournaments/PubMedQA-Normalized-Test" style="color:#7c3aed">PubMedQA</a>:</div>
</div>

In [ ]:
%pip install -q --upgrade gradientsio==0.1.3

import os
import gradientsio as gradients
from IPython.display import HTML, display

os.environ["GRADIENTS_API_KEY"] = "paste-your-api-key-here"

sample = gradients.load_dataset_rows("gradients-io-tournaments/PubMedQA-Normalized-Test")[0]
base_answer = gradients.ModelSampler().generate("Qwen/Qwen2.5-3B", [f"{sample['instruction']}\n\nAnswer:"])[0]

display(HTML(f"""
<div style="border:1px solid #e2e8f0; border-radius:14px; margin:16px 0; overflow:hidden;
            font-family:system-ui,-apple-system,sans-serif; box-shadow:0 1px 3px rgba(0,0,0,.06);">
  <div style="background:linear-gradient(135deg,#1a1a2e,#0f3460); color:#fff;
              padding:18px 22px; font-weight:600; font-size:0.95em; line-height:1.5;">
    {sample['instruction'][:300]}
  </div>
  <div style="display:grid; grid-template-columns:1fr 1fr;">
    <div style="padding:18px 22px; border-right:1px solid #e2e8f0;">
      <div style="font-size:0.7em; text-transform:uppercase; letter-spacing:0.08em;
                  margin-bottom:10px; font-weight:700; color:#6b7280;">Expected Answer</div>
      <p style="margin:0; line-height:1.65; font-size:0.9em;">{sample.get('output','').strip()}</p>
    </div>
    <div style="padding:18px 22px; background:#fef2f2;">
      <div style="font-size:0.7em; text-transform:uppercase; letter-spacing:0.08em;
                  margin-bottom:10px; font-weight:700; color:#dc2626;">Base Model</div>
      <p style="margin:0; line-height:1.65; font-size:0.9em; color:#6b7280;">{base_answer.strip()}</p>
    </div>
  </div>
</div>
"""))

<div style="border-left: 4px solid #7c3aed; padding: 20px 24px; margin: 36px 0 14px 0; background: linear-gradient(90deg, #f5f3ff 0%, #ffffff 100%); border-radius: 0 12px 12px 0; font-family: system-ui, -apple-system, sans-serif;">
  <div style="font-weight: 700; font-size: 1.2em; color: #1a1a2e;">The Fix: One call to train</div>
  <div style="color: #6b7280; font-size: 0.95em; margin-top: 6px; line-height: 1.5;">The base model couldn't reason about medical evidence. To fix that, we point Gradients at <a href="https://huggingface.co/datasets/gradients-io-tournaments/PubMedQA-Normalized-Train" style="color:#7c3aed">PubMedQA</a> &mdash; a dataset of biomedical questions with evidence-based answers. One call. Gradients handles the data prep, GPU scheduling, training, and evaluation.</div>
</div>

In [ ]:
trained_model = gradients.GradientsClient().train(
    model="Qwen/Qwen2.5-3B",
    task_type=gradients.TaskType.INSTRUCT,
    hours=2,
    dataset="gradients-io-tournaments/PubMedQA-Normalized-Train",
    field_instruction="instruction",
    field_input="input",
    field_output="output",
).wait().trained_model_repository

<div style="border-left: 4px solid #7c3aed; padding: 20px 24px; margin: 36px 0 14px 0; background: linear-gradient(90deg, #f5f3ff 0%, #ffffff 100%); border-radius: 0 12px 12px 0; font-family: system-ui, -apple-system, sans-serif;">
  <div style="font-weight: 700; font-size: 1.2em; color: #1a1a2e;">The Proof: Same unseen questions, both models, side by side</div>
  <div style="color: #6b7280; font-size: 0.95em; margin-top: 6px; line-height: 1.5;">Held-out medical questions from PubMedQA that were <em>not</em> in the training set. The original base model and the newly trained model answer each one.</div>
</div>

In [ ]:
samples = gradients.load_dataset_rows("gradients-io-tournaments/PubMedQA-Normalized-Test")
prompts = [f"{r['instruction']}\n\nAnswer:" for r in samples]

sampler = gradients.ModelSampler()
base_answers = sampler.generate("Qwen/Qwen2.5-3B", prompts)
trained_answers = sampler.generate_with_adapter(trained_model, prompts, base_model_repo="Qwen/Qwen2.5-3B")

display(HTML("""
<style>
.g-card { border:1px solid #e2e8f0; border-radius:14px; margin:20px 0; overflow:hidden;
          font-family:system-ui,-apple-system,sans-serif; box-shadow:0 1px 3px rgba(0,0,0,.06); }
.g-card-q { background:linear-gradient(135deg,#1a1a2e,#0f3460); color:#fff;
            padding:18px 22px; font-weight:600; font-size:0.95em; line-height:1.5; }
.g-card-q span { background:rgba(124,58,237,.5); padding:2px 10px; border-radius:20px;
                 font-size:0.8em; margin-right:10px; font-weight:700; }
.g-cols { display:grid; grid-template-columns:1fr 1fr 1fr; }
.g-col { padding:18px 22px; border-right:1px solid #e2e8f0; }
.g-col:last-child { border-right:none; }
.g-lbl { font-size:0.7em; text-transform:uppercase; letter-spacing:0.08em;
         margin-bottom:10px; font-weight:700; }
.g-expected .g-lbl { color:#6b7280; }
.g-trained { background:linear-gradient(180deg,#f5f3ff,#fff); }
.g-trained .g-lbl { color:#7c3aed; }
.g-base { color:#9ca3af; }
.g-base .g-lbl { color:#d1d5db; }
.g-col p { margin:0; line-height:1.65; font-size:0.9em; }
</style>
""" + "\n".join(f"""
<div class="g-card">
  <div class="g-card-q"><span>Q{i}</span>{row['instruction'][:200]}</div>
  <div class="g-cols">
    <div class="g-col g-expected">
      <div class="g-lbl">Expected Answer</div>
      <p>{row.get('output','').strip()}</p>
    </div>
    <div class="g-col g-trained">
      <div class="g-lbl">Trained Model</div>
      <p>{trained.strip()}</p>
    </div>
    <div class="g-col g-base">
      <div class="g-lbl">Base Model</div>
      <p>{base.strip()}</p>
    </div>
  </div>
</div>""" for i, (row, base, trained) in enumerate(zip(samples, base_answers, trained_answers), 1))))

<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); border-radius: 16px; padding: 40px 44px; margin-top: 36px; font-family: system-ui, -apple-system, sans-serif;">
  <h2 style="color: #fff; margin: 0 0 12px 0; font-size: 1.5em;">That's it.</h2>
  <p style="color: #a8b2d1; margin: 0; font-size: 1.05em; line-height: 1.6;">A base model that couldn't handle medical reasoning. One API call to train it on domain data. A fine-tuned model that answers like a specialist.<br><br>Swap PubMedQA for your own dataset &mdash; legal, finance, support, code &mdash; and the workflow is the same.</p>
  <a href="https://www.gradients.io/" style="display: inline-block; margin-top: 20px; padding: 12px 28px; background: #7c3aed; color: #fff; border-radius: 8px; text-decoration: none; font-weight: 600; font-size: 0.95em;">Get started at gradients.io</a>
</div>